# Aşama 1 — Deney Motoru Doğrulama

Bu notebook **tez bulgusu üretmez**. Amaç, büyük aktivasyon–optimizer deneylerinden önce veri hazırlama ve checkpoint/resume altyapısının güvenilirliğini T4 üzerinde doğrulamaktır.

Tez önerisindeki kontrollü karşılaştırma yaklaşımına uygun olarak aynı veri bölünmesi, aynı mimari, aynı eğitim bütçesi ve sabit seed kullanılır.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel public GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests
from pathlib import Path

OWNER = 'erdemci07'
REPO = 'derin-ogrenme-egitim-dinamikleri'
BRANCH = 'main'
repo_dir = Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root = Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)

commit_resp = requests.get(
    f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}',
    timeout=60,
)
commit_resp.raise_for_status()
commit_sha = commit_resp.json()['sha']

archive_resp = requests.get(
    f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}',
    timeout=120,
)
archive_resp.raise_for_status()

with tarfile.open(fileobj=io.BytesIO(archive_resp.content), mode='r:gz') as tar:
    tar.extractall(extract_root)

kokler = [p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler) != 1:
    raise RuntimeError('GitHub arşivi beklenen klasör yapısında değil.')
shutil.move(str(kokler[0]), str(repo_dir))
os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT'] = commit_sha
print('Repo hazır.')
print('Commit:', commit_sha)


## Google Drive veri ve kalıcı deney yolları

CIFAR-10 `MyDrive/tez_datasets` klasöründen okunur. Split, normalizasyon ve doğrulama çıktıları Drive'da kalıcı tutulur.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK = Path('/content/drive/MyDrive')
DRIVE_VERI = DRIVE_KOK / 'tez_datasets'
DRIVE_PROJE = DRIVE_KOK / 'tez_deney_sonuclari'
DRIVE_AYAR = DRIVE_PROJE / 'ortak_ayarlar'
DRIVE_CIKTI = DRIVE_PROJE / 'asama1_deney_motoru_dogrulama'

if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')

DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)

SPLIT_DOSYASI = DRIVE_AYAR / 'cifar10_split.json'
STATS_DOSYASI = DRIVE_AYAR / 'cifar10_stats.json'

os.environ['TEZ_VERI_KLASORU'] = str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI'] = str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI'] = str(STATS_DOSYASI)
os.environ['TEZ_CIKTI_KLASORU'] = str(DRIVE_CIKTI)

print('Veri:', DRIVE_VERI)
print('Split:', SPLIT_DOSYASI)
print('Stats:', STATS_DOSYASI)
print('Çıktı:', DRIVE_CIKTI)


## Bağımlılık ve CUDA kontrolü


In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA aktif değil. T4 GPU seçilmelidir.')


## Kesintisiz çalışma ile checkpoint/resume çalışmasını karşılaştır

Toplam üç epoch çalıştırılır. Önce 3 epoch kesintisiz referans alınır. Sonra aynı koşu 1. epoch sonunda checkpoint alınıp model ve DataLoader yeniden oluşturularak 2. ve 3. epochlardan devam ettirilir.


In [ ]:
import importlib
import dogrula_deney_motoru as dogrulama_modulu
importlib.reload(dogrulama_modulu)

dogrulama_modulu.ana([
    '--veri-klasoru', str(DRIVE_VERI),
    '--split-dosyasi', str(SPLIT_DOSYASI),
    '--stats-dosyasi', str(STATS_DOSYASI),
    '--cikti-klasoru', str(DRIVE_CIKTI),
    '--epoch', '3',
    '--kesinti-epoch', '1',
    '--batch-size', '128',
    '--num-workers', '0',
    '--split-seed', '2026',
    '--seed', '42',
    '--tolerans', '1e-7',
])


## Sonucu özetle


In [ ]:
import json
sonuc_yolu = DRIVE_CIKTI / 'dogrulama_sonucu.json'
sonuc = json.loads(sonuc_yolu.read_text(encoding='utf-8'))

print('GEÇTİ:', sonuc['gecti'])
print('Model hash eşit:', sonuc['karsilastirma']['model_hash_esit'])
print('Maksimum parametre farkı:', sonuc['karsilastirma']['max_parametre_mutlak_farki'])
print('Metrik farkları:', sonuc['karsilastirma']['maksimum_metrik_farklari'])
print('Train / Val:', sonuc['veri_dogrulama']['train_count'], '/', sonuc['veri_dogrulama']['validation_count'])
print('Mean:', sonuc['veri_dogrulama']['normalization_mean'])
print('Std:', sonuc['veri_dogrulama']['normalization_std'])
print('Dosya:', sonuc_yolu)

if not sonuc['gecti']:
    raise RuntimeError('Deney motoru doğrulaması başarısız; pilot başlatılmamalıdır.')
